# 3 . Ejercicio bermuda: Longstaff-Schwartz

Un put bermuda (fechas de ejercicio discretas) se construye con
`qd.exercise(id, dates, exercise_value, continuation)` -- el motor resuelve la politica optima
de ejercicio via regresion de Longstaff-Schwartz sobre las trayectorias Monte Carlo simuladas
bajo `GBM`, y la medida `PayoffExerciseQ` devuelve el precio junto con diagnosticos por fecha
de decision (`.times`=fechas, `.primary`=fraccion de trayectorias ejercidas en esa fecha,
`.secondary`=valor de continuacion condicional). Este notebook visualiza esos diagnosticos y
compara el precio bermuda contra la cota inferior teorica -- el precio europeo equivalente --
y contra su limite cuando el numero de fechas de ejercicio crece (aproximacion al americano).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "../../../build/clients/python")
sys.path.insert(0, "../src")

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import engine
import quantdesk as qd
from quantdesk import greeks

print("modulo engine importado desde:", engine.__file__)


modulo engine importado desde: S:\Projects\engine_quant\clients\python\notebooks\../../../build/clients/python\engine.cp312-win_amd64.pyd


## 1. Mercado y plantillas de contrato


In [2]:
S0, K, R, Q_DIV, SIGMA, T = 100.0, 100.0, 0.04, 0.0, 0.30, 1.0
OBS = "EQ.SPOT.IDX"

model = qd.Gbm(s0=S0, r=R, q=Q_DIV, sigma=SIGMA, observable=OBS)
market = qd.Market(pillars=[T], zero_rates=[R])
eng = qd.Engine(backend="cpu", n_paths=100_000, n_steps=1, seed=41)


def european_put_contract(strike):
    # qd.put_leg (PLAN_IMPROVE_NOTEBOOK2.md Fase 6) en vez de horneado a mano con
    # qd.when/qd.cashflow/qd.maximum -- devuelve el mismo Contract crudo (qty=1.0, sin envolver
    # en PayoffProduct), que es exactamente lo que necesita bermudan_put_contract como rama de
    # continuacion.
    return qd.put_leg(OBS, strike, 1.0, T)


def bermudan_put_contract(strike, exercise_dates):
    exercise_value = qd.maximum(strike - qd.current(OBS), 0.0)
    continuation = european_put_contract(strike)
    return qd.exercise("EX", exercise_dates, exercise_value, continuation)


def price_european_put(strike):
    trade = qd.PayoffProduct(id="EU", contract=european_put_contract(strike))
    return eng.price(trade, model, market, ["PayoffPriceQ"]).PayoffPriceQ.scalar


def price_bermudan_put(strike, exercise_dates):
    trade = qd.PayoffProduct(id="BM", contract=bermudan_put_contract(strike, exercise_dates))
    return eng.price(trade, model, market, ["PayoffExerciseQ"]).PayoffExerciseQ


print(f"S0={S0}  K={K}  sigma={SIGMA:.0%}  T={T}y")

S0=100.0  K=100.0  sigma=30%  T=1.0y


## 2. Put bermuda vs put europea (misma cesta de parametros)


In [3]:
EXERCISE_DATES = [0.25, 0.5, 0.75]

european_price = price_european_put(K)
bermudan_result = price_bermudan_put(K, EXERCISE_DATES)
bermudan_price = bermudan_result.scalar

fig = go.Figure(
    go.Bar(
        x=["Europea", "Bermuda (3 fechas)"],
        y=[european_price, bermudan_price],
        marker_color=["#1f77b4", "#ff7f0e"],
        text=[f"{european_price:.3f}", f"{bermudan_price:.3f}"],
        textposition="outside",
    )
)
fig.update_layout(
    title="El derecho de ejercicio anticipado tiene valor: bermuda >= europea",
    yaxis_title="precio",
    template="plotly_white",
)
fig.show()

print(f"europea={european_price:.4f}  bermuda={bermudan_price:.4f}  prima de ejercicio anticipado={bermudan_price - european_price:.4f}")


europea=9.8758  bermuda=10.1203  prima de ejercicio anticipado=0.2444


## 3. Diagnosticos de ejercicio por fecha de decision

`PayoffExerciseQ.primary` es la fraccion de trayectorias que, en cada fecha de decision,
la politica de Longstaff-Schwartz decide ejercer (en vez de continuar) -- `.secondary` es el
valor de continuacion condicional medio en esa fecha.


In [4]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Decision de ejercicio por fecha", "Valor de continuacion en cada fecha de decision"))

fig.add_trace(
    go.Bar(x=[f"t={t}" for t in bermudan_result.times], y=bermudan_result.primary, marker_color="#ff7f0e", showlegend=False),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=bermudan_result.times, y=bermudan_result.secondary, mode="lines+markers", line=dict(color="seagreen"), showlegend=False),
    row=1,
    col=2,
)

fig.update_yaxes(title_text="fraccion de trayectorias ejercidas", row=1, col=1)
fig.update_yaxes(title_text="valor de continuacion condicional", row=1, col=2)
fig.update_xaxes(title_text="tiempo", row=1, col=2)
fig.update_layout(template="plotly_white", width=1100, height=450)
fig.show()

for t, frac, cont in zip(bermudan_result.times, bermudan_result.primary, bermudan_result.secondary):
    print(f"t={t:.2f}  ejercida={frac:.1%}  continuacion={cont:,.1f}")


t=0.25  ejercida=6.5%  continuacion=50,562.0
t=0.50  ejercida=29.0%  continuacion=50,418.0
t=0.75  ejercida=46.7%  continuacion=50,548.0


## 4. Prima de ejercicio anticipado frente a moneyness


In [5]:
strikes = np.arange(70.0, 131.0, 10.0)
european_curve, bermudan_curve = [], []
for k in strikes:
    european_curve.append(price_european_put(float(k)))
    bermudan_curve.append(price_bermudan_put(float(k), EXERCISE_DATES).scalar)

european_curve = np.array(european_curve)
bermudan_curve = np.array(bermudan_curve)

fig = go.Figure()
fig.add_trace(go.Scatter(x=strikes, y=european_curve, mode="lines+markers", name="Europea", line=dict(color="royalblue")))
fig.add_trace(go.Scatter(x=strikes, y=bermudan_curve, mode="lines+markers", name="Bermuda (3 fechas)", line=dict(color="darkorange")))
fig.add_trace(
    go.Scatter(
        x=np.concatenate([strikes, strikes[::-1]]),
        y=np.concatenate([bermudan_curve, european_curve[::-1]]),
        fill="toself",
        fillcolor="rgba(128,128,128,0.25)",
        line=dict(color="rgba(255,255,255,0)"),
        name="prima de ejercicio anticipado",
        hoverinfo="skip",
    )
)
fig.update_layout(
    title="Put bermuda vs europea a lo largo del strike",
    xaxis_title="strike",
    yaxis_title="precio",
    template="plotly_white",
)
fig.show()


## 5. Convergencia hacia el limite americano

Cuantas mas fechas de ejercicio se permiten, mas se acerca el precio bermuda al limite
americano (ejercicio continuo). El ruido Monte Carlo de la regresion de Longstaff-Schwartz es
visible en la curva -- coherente con que cada punto es una regresion independiente sobre un
numero finito de trayectorias, no una formula cerrada.


In [6]:
n_dates_grid = [1, 2, 4, 6, 12, 24]
convergence_prices = []
for n in n_dates_grid:
    dates = list(np.linspace(T / (n + 1), T, n + 1))[:-1]
    convergence_prices.append(price_bermudan_put(K, dates).scalar)

fig = go.Figure()
fig.add_trace(go.Scatter(x=n_dates_grid, y=convergence_prices, mode="lines+markers", name="bermuda", line=dict(color="firebrick")))
fig.add_hline(y=european_price, line=dict(color="royalblue", dash="dash"), annotation_text="europea (1 fecha, limite inferior)")
fig.update_layout(
    title="Bermuda -> americano: mas fechas de decision, mas se acerca al limite continuo",
    xaxis_title="numero de fechas de ejercicio antes de vencimiento",
    yaxis_title="precio",
    template="plotly_white",
)
fig.show()
